# 06 · Which quantity should drive a Lanchester fit?

Classical Lanchester theory (1916) models **unit counts**, not aggregate health: `dB/dt = -a·R(t)`,
`dR/dt = -b·B(t)`. The mechanism is each side's *currently alive* shooters contributing roughly
constant firepower until they die — which matches SMAClite's actual damage model (a unit deals
full damage every attack regardless of its own remaining HP, right up until it hits zero) better
than health does. Health is a looser continuous proxy that happens to correlate with unit count.

But health is a much *smoother* signal to differentiate than unit count (see the project's
`docs/` notes on messy derivatives), and `states.aggregate.engagement` now gives a third,
arguably even more mechanistically faithful candidate: `engaged_*` counts only units *currently
in range*, since a unit that's alive but still approaching contributes zero current firepower,
same as a dead one.

This notebook fits the **exact** classical square law (not a general SINDy fit that happens to
sparsify to that form — `models.sindy.fit_lanchester` is a fixed 2-parameter model, no constant,
no self-term) on all three candidate drivers — `health`, `alive`, `engaged` — on the same episodes,
and checks each one's sensitivity to the smoothing window, the way notebook 04's Part 4 did for
the general SINDy fit.

Uses `exp03` (`10m_vs_10m`, randomized spawn) because it's the only experiment collected with the
`vx`/`vy`/`target_*` snapshot columns `engagement()` needs — `exp01`/`exp02` predate them.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sindy_wm.data import storage
from sindy_wm.models.sindy import fit_lanchester, plot_predictions, summarize_eval, to_trajectory
from sindy_wm.paths import RAW_DIR
from sindy_wm.states.aggregate import engagement, level_a

plt.rcParams["figure.dpi"] = 100

EXPERIMENT_DIR = RAW_DIR / "exp03_10m_random_aggression_stepmul"
DATASET = "10m_attack_a070_s1_v1"  # balanced: win rate 0.38, 479 distinct outcomes
N_UNITS = 10  # per side, for normalizing engaged_* onto the same 0-1-ish scale as health/alive

DOWNSAMPLE = 2
SMOOTH_WINDOW = 9
TEST_FRACTION = 0.2
SPLIT_SEED = 0

## 1 · Load Level A + engagement, joined

`engaged_*`/`targets_*`/`speed_*` are computed per episode the same way `level_a` is, then joined
on the shared `(episode_idx, step)` index. `engaged_*` is divided by `N_UNITS` so its scale matches
`health`/`alive` (both already normalized to start at 1), making RMSE comparable across drivers.

In [ ]:
dataset_dir = EXPERIMENT_DIR / DATASET
manifest = storage.load_manifest(dataset_dir)
index = storage.load_episode_index(dataset_dir).set_index("episode_idx")
dt = manifest["config"]["env"]["seconds_per_step"]

snaps = storage.load_snapshots(dataset_dir)
lvl_a = snaps.groupby("episode_idx").apply(level_a, normalize=True, include_groups=False)
eng = snaps.groupby("episode_idx").apply(engagement, include_groups=False)
eng[["engaged_blue", "engaged_red"]] = eng[["engaged_blue", "engaged_red"]] / N_UNITS
states = lvl_a.join(eng)
states["t"] = states.index.get_level_values("step") * dt

print(f"{DATASET}: {len(index)} episodes, Blue win rate {index['battle_won'].mean():.0%}")
states.head()

## 2 · Trajectories: trimmed once via health, shared across all three drivers

`to_trajectory` trims rows, not columns — trimming by `health_blue`/`health_red` (the signal this
project already trusts for finding the start of engagement) still leaves `alive_*`/`engaged_*`
columns in the result. All three drivers below are therefore evaluated on *exactly* the same
time window per episode — only which columns get fit differs, not how much of the episode is used.

In [ ]:
trajectories = {
    i: to_trajectory(states.loc[i], ["health_blue", "health_red"], True, DOWNSAMPLE) for i in index.index
}

rng = np.random.default_rng(SPLIT_SEED)
episode_ids = rng.permutation(index.index.to_numpy())
n_test = int(round(TEST_FRACTION * len(episode_ids)))
test_ids = sorted(episode_ids[:n_test].tolist())
train_ids = sorted(episode_ids[n_test:].tolist())
dt_fit = dt * DOWNSAMPLE

lengths = pd.Series({i: len(trajectories[i]) for i in index.index})
print(f"{len(train_ids)} training and {len(test_ids)} test episodes, dt = {dt_fit} s")
print(f"Points per trajectory: mean {lengths.mean():.0f}, min {lengths.min()}, max {lengths.max()}")

DRIVERS = {
    "health": ["health_blue", "health_red"],
    "alive": ["alive_blue", "alive_red"],
    "engaged": ["engaged_blue", "engaged_red"],
}

## 3 · Fit the classical square law on each driver

Same `SmoothedFiniteDifference` derivative estimate for all three (so differences in the result
come from the driver, not from a different differentiation setup).

In [ ]:
models = {}
results = []
for name, cols in DRIVERS.items():
    m = fit_lanchester(trajectories, train_ids, cols, dt_fit, SMOOTH_WINDOW)
    metrics = summarize_eval(m, test_ids, trajectories, cols, index["battle_won"])
    models[name] = m
    results.append({"driver": name, "a": m.a, "b": m.b, **metrics})
    print(f"{name:>8}: (x1)' = {m.equations(precision=4)[0]}   (x2)' = {m.equations(precision=4)[1]}")

results = pd.DataFrame(results).set_index("driver")
results[["a", "b", "rmse_midway", "winner_acc_midway", "leader_baseline_midway", "failed_simulations"]]

## 4 · Is each driver's fit actually trustworthy, or just a lucky `SMOOTH_WINDOW`?

The same check as notebook 04's Part 4, now run once per driver. `alive`/`engaged` change far
less often than `health` (one event per unit death, not per shot), so they're the more likely
candidates to be sensitive to the window — this checks that directly instead of assuming it.

In [ ]:
sweep = []
for name, cols in DRIVERS.items():
    for window in [5, 9, 13, 17]:
        m = fit_lanchester(trajectories, train_ids, cols, dt_fit, window)
        metrics = summarize_eval(m, test_ids, trajectories, cols, index["battle_won"])
        sweep.append({"driver": name, "smooth_window": window, "a": m.a, "b": m.b, **metrics})
sweep = pd.DataFrame(sweep)
sweep.pivot(index="smooth_window", columns="driver", values=["a", "b"])

**What to look for:** `health`'s `a`/`b` should barely move (notebook 04 Part 4 already found this).
If `alive`'s or `engaged`'s coefficients swing a lot across the four windows, don't trust the
single-window fit above for that driver — it's in the "messy derivative" regime this project keeps
checking for, not a stable discovered law.

## 5 · Simulated vs. actual, per driver

One won and one lost test episode, for each driver's chosen (`SMOOTH_WINDOW`) fit.

In [ ]:
test_won = [i for i in test_ids if index.loc[i, "battle_won"]][:1]
test_lost = [i for i in test_ids if not index.loc[i, "battle_won"]][:1]

for name, cols in DRIVERS.items():
    plot_predictions(
        models[name],
        test_won + test_lost,
        trajectories,
        cols,
        index["battle_won"],
        title=f"{name} (a={models[name].a:.3f}, b={models[name].b:.3f})",
    )
    plt.show()

## 6 · Reading the result

A property worth noticing before concluding anything: with `a, b > 0`, the classical square law's
own structure — both quantities only ever decreasing, each faster when the opponent is currently
stronger — makes a lead reversal from the midpoint onward mathematically rare. So `winner_acc_midway`
landing close to `leader_baseline_midway` for a *given* driver isn't necessarily a coincidence: it's
close to what the model's functional form can do by construction, for any driver. The real
differences between drivers show up in `rmse_midway` (how well the *trajectory*, not just the
eventual winner, is tracked) and in whether `a`/`b` survive the Part 4-style sweep in section 4.

Compare the chosen driver's `winner_acc_midway` here against the **general** (non-Lanchester-
constrained) SINDy fit on `health` in notebook 04 — that one is not restricted to pure cross-
coupling and could pick up a constant term, self-terms, and (at `degree=2`) quadratic terms.
Whether that extra freedom actually earns it a real improvement over both the classical square law
*and* `leader_baseline_midway`, rather than just overfitting, is the real test of whether SINDy is
finding something beyond classical Lanchester theory here.

## Next steps

- Run this on more `exp03` settings (not just one balanced dataset) to see if the driver ranking
  holds generally, or is specific to this aggression/step_mul combination.
- Re-collect a 5v5 dataset with the new snapshot columns, to compare `alive`/`engaged` driver
  behavior across scenario sizes — classical theory predicts `a`/`b` should be roughly invariant
  to force size under the square law; this notebook only has one scale to look at so far.
- If `engaged`'s coefficients turn out to be stable-but-wrong-signed (not just noisy) across more
  settings, that's evidence it's the wrong *kind* of variable for this functional form — a
  fluctuating activity measure, not a monotonically depleting stock — not just under-smoothed.